In [1]:
import os

In [2]:
from scipy import io
import os

resnet_101_path = os.path.join(
    "datasets", "AwA2", "xlsa17", "data", "AWA2", "res101.mat"
)

data = io.loadmat(resnet_101_path)

print(data.keys())

print(data["image_files"].shape)

dict_keys(['__header__', '__version__', '__globals__', 'features', 'image_files', 'labels'])
(37322, 1)


# AwA2 incomplete concept sets

Creates a reduced AwA2 predicate matrix (the "incomplete concept set") that a run
consumes via `incomplete=True data.pkl_file_dir=<folder>/`.

The run reads exactly one file from the folder: `predicate-matrix-binary.txt`.
Everything else written alongside it (`info.json`, `concept_names.txt`,
`removed_concepts.npy`, `concept_groups.json`) is for post-hoc analysis.

Nothing here touches the image split — `{train,val,test}_split.npz` is untouched, so a
complete and an incomplete run see exactly the same images and class labels.

In [3]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "datasets" / "awa2_dataset.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from types import SimpleNamespace
import numpy as np

from datasets.awa2_dataset import (
    CONCEPT_SEMANTICS,
    CONCEPT_GROUPS,
    N_CONCEPTS,
    create_custom_incomplete_dataset,
)

# Same rule train.py's update_config_paths applies, so the notebook writes where the
# run will look. On the cluster that is the forced path; locally it is the repo copy.
DEFAULT_DATA_PATH = (
    "/cluster/home/smarcou/work/Data/"
    if "biomed" in os.uname()[1]
    else str(ROOT / "datasets") + "/"
)
print("data_path:", DEFAULT_DATA_PATH)

data_path: /Users/stephenmarcou/Documents/ETH Zurich/Cambridge/Code/SCBM_implementation/datasets/


## The function

`keep` accepts **concept names** (from `CONCEPT_SEMANTICS`) and/or **group names** (from
`CONCEPT_GROUPS`) interchangeably — a group name expands to all of its concepts. Order
and duplicates do not matter; the saved matrix always uses the original AwA2 concept
ordering, which is what the old→new index mapping in `info.json` is keyed on.

In [4]:
def make_incomplete_awa2(keep, data_path=None, incomplete_dir="incomplete_data/"):
    """Create an incomplete AwA2 concept set that keeps `keep`.

    Parameters
    ----------
    keep : iterable of str
        Concept names from CONCEPT_SEMANTICS and/or group names from CONCEPT_GROUPS.
        Group names expand to all concepts in the group.
    data_path : str, optional
        Root holding AwA2/. Defaults to DEFAULT_DATA_PATH.
    incomplete_dir : str
        Subfolder of AwA2/ holding concept sets. Must match config.data.incomplete_dir.

    Returns
    -------
    (folder_name, num_concepts)
        folder_name is what you pass as data.pkl_file_dir=<folder_name>.
    """
    keep = list(keep)
    if not keep:
        raise ValueError("keep is empty; an incomplete set must retain at least one concept.")

    concept_set = set(CONCEPT_SEMANTICS)
    unknown = [k for k in keep if k not in concept_set and k not in CONCEPT_GROUPS]
    if unknown:
        raise ValueError(
            f"Not a concept or group name: {unknown}. "
            f"Concepts: {sorted(concept_set)[:5]}... Groups: {list(CONCEPT_GROUPS)}"
        )

    # Expand group names, then restore original AwA2 ordering.
    keep_idx = set()
    for name in keep:
        if name in CONCEPT_GROUPS:
            keep_idx.update(CONCEPT_GROUPS[name])
        else:
            keep_idx.add(CONCEPT_SEMANTICS.index(name))
    keep_names = [CONCEPT_SEMANTICS[i] for i in sorted(keep_idx)]

    if len(keep_names) == N_CONCEPTS:
        raise ValueError("keep covers all 85 concepts; that is the complete set, not an incomplete one.")

    hidden = [c for c in CONCEPT_SEMANTICS if c not in set(keep_names)]
    print(f"keeping {len(keep_names)} concepts, hiding {len(hidden)}: {hidden}\n")

    cfg = SimpleNamespace(
        data_path=data_path or DEFAULT_DATA_PATH,
        incomplete_dir=incomplete_dir,
    )
    folder, n = create_custom_incomplete_dataset(cfg, keep_names)

    print(f"\n  data.pkl_file_dir={folder}")
    print(f"  rsync -av {os.path.join(cfg.data_path, 'AwA2', incomplete_dir, folder.rstrip('/'))} \\")
    print(f"    smarcou@<cluster>:/cluster/home/smarcou/work/Data/AwA2/incomplete_data/")
    return folder, n

In [5]:
keep = ["black", "gray", "stripes", "hairless", "flippers", "paws", "plains", "fierce", "solitary"]

#make_incomplete_awa2(keep, data_path=DEFAULT_DATA_PATH, incomplete_dir="incomplete_data/")


## Reference: the 28 semantic groups

In [6]:
for g, idx in CONCEPT_GROUPS.items():
    print(f"{g:22s} {len(idx):3d}  {', '.join(CONCEPT_SEMANTICS[i] for i in idx)}")

color                    8  black, white, blue, brown, gray, orange, red, yellow
fur_pattern              6  patches, spots, stripes, furry, hairless, toughskin
size                     4  big, small, bulbous, lean
limb_shape               7  flippers, hands, hooves, pads, paws, longleg, longneck
tail                     1  tail
teeth_type               4  chewteeth, meatteeth, buckteeth, strainteeth
horns                    1  horns
claws                    1  claws
tusks                    1  tusks
smelly                   1  smelly
transport_mechanism      5  flys, hops, swims, tunnels, walks
speed                    2  fast, slow
strength                 2  strong, weak
muscle                   1  muscle
movement_move            2  bipedal, quadrapedal
active                   2  active, inactive
nocturnal                1  nocturnal
hibernate                1  hibernate
agility                  1  agility
diet                     5  fish, meat, plankton, vegetation, insects
feedin

## Usage

Keep an explicit list of concepts:

```python
folder, n = make_incomplete_awa2(["black", "white", "big", "small", "tail", "claws"])
```

Keep whole groups by name:

```python
folder, n = make_incomplete_awa2(["color", "size", "fur_pattern", "limb_shape"])
```

Hide a group (keep everything else) — the complement idiom:

```python
HIDE = {"biome"}
folder, n = make_incomplete_awa2([g for g in CONCEPT_GROUPS if g not in HIDE])
```

In [7]:
# folder, n = make_incomplete_awa2([g for g in CONCEPT_GROUPS if g not in {"biome"}])

## Verify

The run derives `data.num_concepts` from this matrix's column count, so this is the
whole run-side contract: 50 rows, `n` columns.

In [8]:
def check_incomplete_awa2(folder, data_path=None, incomplete_dir="incomplete_data/"):
    root = os.path.join(data_path or DEFAULT_DATA_PATH, "AwA2", incomplete_dir, folder.rstrip("/"))
    M = np.genfromtxt(os.path.join(root, "predicate-matrix-binary.txt"), dtype=int)
    removed = np.load(os.path.join(root, "removed_concepts.npy"))
    print(f"{root}\n  matrix {M.shape}  ->  data.num_concepts={M.shape[1]}")
    print(f"  hidden ({len(removed)}): {[CONCEPT_SEMANTICS[i] for i in removed]}")
    return M

# check_incomplete_awa2(folder)

## Intervention curves

Reads `intervention_log.txt` from each run under `experiments/<model_type>/AwA2/` and
plots task accuracy vs. number of concepts intervened on. Same parsing idiom as
`waterbirds_analysis.ipynb` (`parse_intervention_log` / `collect_runs`).

In [9]:
import re
import ast
import matplotlib.pyplot as plt

# (label, model_type, run_dir) -- model_type selects the experiments/<model_type>/AwA2/
# subfolder. Listed explicitly so that adding new runs to experiments/ does not silently
# change this plot; add a line here to include a run.
# All four share pkl_file_dir=awa2_incomplete_local_custom_1/ (9 concepts kept, 50 classes).
# Caveat on CEM: it is the only run trained on interventions (RandInt, model.p_int=0.25),
# so its slope with concept count is not a like-for-like comparison against the others.
# The hard CBM is a plain baseline (concept_learning="hard", no training-time interventions).
AWA2_RUNS = [
    ("SCBM", "scbm", "incomplete_awa2_2026-09-08_15-46-51_2e39d"),
    ("Res-SCBM (20 res), L_int_ext w=1", "scbm_residual",
     "incomplete_awa2_emp_perc_residuals_20_L_int_extension_loss_weight_1_2026-09-08_12-27-06_91dca"),
    ("Res-SCBM (20 res), no L_int_ext", "scbm_residual",
     "incomplete_awa2_emp_perc_residuals_20_2026-09-08_22-24-14_ccfa3"),
    ("CEM (RandInt p_int=0.25)", "cbm",
     "cem_incomplete_awa2_pint_025_2026-09-09_12-24-08_b8349"),
    ("Hard CBM", "cbm",
     "hard_cbm_incomplete_awa2_2026-09-09_13-09-42_1eccd"),
]

COLORS = {
    "SCBM": "#2a78d6",
    "Res-SCBM (20 res), L_int_ext w=1": "#1baf7a",
    "Res-SCBM (20 res), no L_int_ext": "#e07b39",
    "CEM (RandInt p_int=0.25)": "#8e5ea2",
    "Hard CBM": "#c9464b",
}
LINESTYLES = {"cbm": "--", "cbm_residual": "--"}


def read_run_config(run_path):
    """First line of log.txt is the full config dict."""
    with open(os.path.join(run_path, "log.txt"), "r") as f:
        return ast.literal_eval(f.readline().strip())


def parse_intervention_log(log_txt_path):
    """Return (xs, rows) with rows[i] a {metric: value} dict for xs[i] concepts intervened.

    Keyed by concept count so that a log containing a partial sweep followed by a
    re-run keeps the *last* value written.
    """
    curve = {}
    with open(log_txt_path, "r") as f:
        for line in f:
            m = re.search(r"Intervention on (\d+) concepts", line)
            if not m or "y_accuracy" not in line:
                continue
            metrics = {
                k: float(v) for k, v in re.findall(r"([A-Za-z_\-]+):\s*([0-9.]+)", line)
            }
            curve[int(m.group(1))] = metrics
    xs = sorted(curve)
    return xs, [curve[x] for x in xs]


def collect_runs(runs, dataset, metric="y_accuracy"):
    collected = []
    for label, model_type, run_dir in runs:
        run_path = os.path.join("experiments", model_type, dataset, run_dir)
        cfg = read_run_config(run_path)
        xs, rows = parse_intervention_log(os.path.join(run_path, "intervention_log.txt"))
        collected.append({
            "label": label,
            "run_dir": run_dir,
            "model_type": model_type,
            "dataset": dataset,
            "num_classes": cfg["data"]["num_classes"],
            "num_concepts": cfg["data"]["num_concepts"],
            "x": xs,
            "y": [r[metric] * 100 for r in rows],
        })
    return collected


metric = "y_accuracy"
runs = collect_runs(AWA2_RUNS, "AwA2", metric=metric)
for r in runs:
    print(f"[{r['dataset']} {r['num_classes']} classes, {r['num_concepts']} concepts] "
          f"{r['label']:<40} {len(r['x']):>2} points  {r['run_dir']}")

# The title reports runs[0]'s shape, so make sure every run really is on the same setup.
assert len({(r["num_classes"], r["num_concepts"]) for r in runs}) == 1, \
    "runs differ in num_classes/num_concepts; they are not comparable on one axis"

fig, ax = plt.subplots(figsize=(7, 5))
for r in runs:
    ax.plot(
        r["x"], r["y"], marker="o", ms=4,
        color=COLORS.get(r["label"]),
        linestyle=LINESTYLES.get(r["model_type"], "-"),
        label=r["label"],
    )
ax.set_title(f"AwA2 — {runs[0]['num_classes']}-class target, {runs[0]['num_concepts']} concepts kept")
ax.set_xlabel("Number of concepts intervened on")
ax.set_ylabel("Task accuracy (%)")
lo = min(min(r["y"]) for r in runs)
ax.set_ylim(max(0, lo - 10), 100)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'experiments/scbm/AwA2/incomplete_awa2_2026-09-08_15-46-51_2e39d/log.txt'

In [ ]:
# Training length per run: the configured budget (model.j_epochs, all four are
# training_mode="joint") next to the last epoch the log actually reached, since early
# stopping (patience=30) means the budget is an upper bound, not what was run.
def read_epoch_budget(run_path):
    """(config, last epoch index seen in log.txt) for one run."""
    cfg = read_run_config(run_path)
    last = -1
    with open(os.path.join(run_path, "log.txt"), "r") as f:
        next(f)  # config line
        for line in f:
            m = re.match(r"Epoch (\d+),", line)
            if m:
                last = max(last, int(m.group(1)))
    return cfg, last


print(f"{'run':<40} {'mode':<10} {'j_epochs':>8} {'c/t':>8} {'patience':>8} {'last epoch':>10}")
for label, model_type, run_dir in AWA2_RUNS:
    cfg, last = read_epoch_budget(os.path.join("experiments", model_type, "AwA2", run_dir))
    m = cfg["model"]
    print(f"{label:<40} {m['training_mode']:<10} {m['j_epochs']:>8} "
          f"{str(m['c_epochs']) + '/' + str(m['t_epochs']):>8} "
          f"{m['early_stopping_patience']:>8} {last:>10}")


run                                      mode       j_epochs      c/t patience last epoch
SCBM                                     joint           200  200/100       30        194
Res-SCBM (20 res), L_int_ext w=1         joint           200  200/100       30        170
Res-SCBM (20 res), no L_int_ext          joint           200  200/100       30        151
CEM (RandInt p_int=0.25)                 joint           200  200/100       30         54
Hard CBM                                 joint           200  200/100       30        200


## AwA2 complete: concept-wise vs. group-wise interventions

Complete AwA2 (85 concepts, 50 classes), three seeds per model. Each run has two
intervention curves on the test split, from two `inference.py` passes on the same checkpoint:

- `intervention_log.txt` — policy `random`: every sample gets one more random concept per
  step, 85 steps.
- `intervention_log_random_group.txt` — policy `random_group`
  (`inference.inter_policy=random_group`): every step adds one complete CEM concept group,
  28 steps. The group is drawn per *batch* (the SCBM strategies need all samples in a batch to
  have the same number of intervened concepts), so each batch follows its own random group
  order and the curve averages over batches. Each log line also records the mean number of
  concepts the step amounts to, which the third panel uses to put both policies on one axis.

Bands are ± one std over seeds. Caveat as above: CEM is trained with RandInt
(`model.p_int=0.25`), the others never see interventions during training.


In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt

# Complete AwA2 (85 concepts, 50 classes), three seeds (0, 10, 20) per model. Every run has
# both curves: intervention_log.txt (policy 'random': one concept per sample and step) and
# intervention_log_random_group.txt (policy 'random_group': one of the 28 CEM concept groups
# per batch and step). Listed explicitly, like AWA2_RUNS above.
AWA2_COMPLETE_RUNS = {
    "SCBM": ("scbm", [
        "complete_awa2_2026-09-13_14-47-46_seed_0_664d9",
        "complete_awa2_2026-09-13_16-16-04_seed_10_5363c",
        "complete_awa2_2026-09-13_21-13-42_seed_20_a196d",
    ]),
    "Res-SCBM (20 res), L_int_ext w=1": ("scbm_residual", [
        "complete_awa2_emp_perc_residuals_20_L_int_extension_loss_weight_1_2026-09-13_14-47-46_seed_0_2e046",
        "complete_awa2_emp_perc_residuals_20_L_int_extension_loss_weight_1_2026-09-13_16-15-17_seed_10_d95d1",
        "complete_awa2_emp_perc_residuals_20_L_int_extension_loss_weight_1_2026-09-13_21-06-35_seed_20_2c26f",
    ]),
    "Res-SCBM (20 res), no L_int_ext": ("scbm_residual", [
        "complete_awa2_emp_perc_residuals_20_2026-09-13_14-47-46_seed_0_165a6",
        "complete_awa2_emp_perc_residuals_20_2026-09-13_16-16-04_seed_10_6ac6a",
        "complete_awa2_emp_perc_residuals_20_2026-09-13_21-13-12_seed_20_33c3c",
    ]),
    "CEM (RandInt p_int=0.25)": ("cbm", [
        "cem_p_int_0.25_complete_awa2_2026-09-13_14-47-46_seed_0_908b6",
        "cem_p_int_0.25_complete_awa2_2026-09-13_16-16-30_seed_10_28a8b",
        "cem_p_int_0.25_complete_awa2_2026-09-13_21-13-16_seed_20_e43f9",
    ]),
    "Hard CBM": ("cbm", [
        "hard_cbm_complete_awa2_2026-09-13_14-47-52_seed_0_25c23",
        "hard_cbm_complete_awa2_2026-09-13_16-16-30_seed_10_b6b89",
        "hard_cbm_complete_awa2_2026-09-13_21-26-11_seed_20_8678d",
    ]),
}
COMPLETE_COLORS = {
    "SCBM": "#2a78d6",
    "Res-SCBM (20 res), L_int_ext w=1": "#1baf7a",
    "Res-SCBM (20 res), no L_int_ext": "#e07b39",
    "CEM (RandInt p_int=0.25)": "#8e5ea2",
    "Hard CBM": "#c9464b",
}
INTERVENTION_LOGS = {"concepts": "intervention_log.txt", "groups": "intervention_log_random_group.txt"}

_STEP_RE = re.compile(r"Intervention on (\d+) (concepts|groups)(?: \(mean ([0-9.]+) concepts\))?: ")


def parse_intervention_log_units(log_txt_path):
    """Parse either curve format.

    Returns (unit, xs, mean_concepts, rows): xs are the step counts in `unit` ('concepts'
    or 'groups'), mean_concepts[i] is the mean number of concepts intervened on at xs[i]
    (equal to xs[i] for the concept-wise curve; logged per step for the group-wise one),
    rows[i] the {metric: value} dict. Keyed by step so a partial sweep followed by a re-run
    keeps the last value written.
    """
    curve, unit = {}, None
    with open(log_txt_path, "r") as f:
        for line in f:
            m = _STEP_RE.match(line)
            if not m or "y_accuracy" not in line:
                continue
            step, unit = int(m.group(1)), m.group(2)
            mean_c = float(m.group(3)) if m.group(3) is not None else float(step)
            metrics = {k: float(v) for k, v in re.findall(r"([A-Za-z_\-]+):\s*([0-9.]+)", line[m.end():])}
            curve[step] = (mean_c, metrics)
    xs = sorted(curve)
    return unit, xs, [curve[x][0] for x in xs], [curve[x][1] for x in xs]


def collect_seeds(model_type, run_dirs, log_name, metric="y_accuracy"):
    """Stack one curve over seeds -> dict with x, mean_concepts, y_mean, y_std (in %)."""
    ys, xs_ref, mean_c_all = [], None, []
    for run_dir in run_dirs:
        unit, xs, mean_c, rows = parse_intervention_log_units(
            os.path.join("experiments", model_type, "AwA2", run_dir, log_name)
        )
        if xs_ref is None:
            xs_ref = xs
        assert xs == xs_ref, f"{run_dir}: {log_name} has steps {xs[:3]}..{xs[-1]} vs {xs_ref[:3]}..{xs_ref[-1]}"
        ys.append([r[metric] * 100 for r in rows])
        mean_c_all.append(mean_c)
    ys = np.asarray(ys)
    return {
        "unit": unit, "x": np.asarray(xs_ref), "mean_concepts": np.asarray(mean_c_all).mean(0),
        "y_mean": ys.mean(0), "y_std": ys.std(0), "num_seeds": len(run_dirs),
    }


metric = "y_accuracy"
curves = {
    label: {unit: collect_seeds(model_type, run_dirs, log_name, metric=metric)
            for unit, log_name in INTERVENTION_LOGS.items()}
    for label, (model_type, run_dirs) in AWA2_COMPLETE_RUNS.items()
}

# Summary: accuracy at no / half / full intervention, per curve (mean over seeds).
print(f"{'model':<34} {'seeds':>5} | {'concept-wise: 0':>15} {'~43':>6} {'85':>6} | {'group-wise: 0':>13} {'14':>6} {'28':>6}")
for label, c in curves.items():
    cw, gw = c["concepts"], c["groups"]
    ci = [int(np.argmin(np.abs(cw["x"] - t))) for t in (0, 43, 85)]
    gi = [int(np.argmin(np.abs(gw["x"] - t))) for t in (0, 14, 28)]
    print(f"{label:<34} {cw['num_seeds']:>5} | {cw['y_mean'][ci[0]]:>15.1f} {cw['y_mean'][ci[1]]:>6.1f} {cw['y_mean'][ci[2]]:>6.1f} "
          f"| {gw['y_mean'][gi[0]]:>13.1f} {gw['y_mean'][gi[1]]:>6.1f} {gw['y_mean'][gi[2]]:>6.1f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ax_c, ax_g, ax_both = axes
for label, c in curves.items():
    color = COMPLETE_COLORS.get(label)
    cw, gw = c["concepts"], c["groups"]
    for ax, cur in ((ax_c, cw), (ax_g, gw)):
        ax.plot(cur["x"], cur["y_mean"], color=color, marker="o", ms=2.5, lw=1.4, label=label)
        ax.fill_between(cur["x"], cur["y_mean"] - cur["y_std"], cur["y_mean"] + cur["y_std"], color=color, alpha=0.15, lw=0)
    # Same concept axis for both policies: the group-wise curve sits at the mean number of
    # concepts its k groups amount to.
    ax_both.plot(cw["x"], cw["y_mean"], color=color, lw=1.4, label=f"{label} — concept-wise")
    ax_both.plot(gw["mean_concepts"], gw["y_mean"], color=color, lw=1.4, ls="--", marker="s", ms=3, label=f"{label} — group-wise")

n_seeds = next(iter(curves.values()))["concepts"]["num_seeds"]
ax_c.set_title(f"Concept-wise (policy 'random'), mean ± std over {n_seeds} seeds")
ax_c.set_xlabel("Number of concepts intervened on")
ax_g.set_title(f"Group-wise (policy 'random_group'), mean ± std over {n_seeds} seeds")
ax_g.set_xlabel("Number of concept groups intervened on (of 28)")
ax_both.set_title("Both on the concept axis (group-wise at its mean concept count)")
ax_both.set_xlabel("Number of concepts intervened on")
lo = min(min(cur["y_mean"].min() for cur in c.values()) for c in curves.values())
for ax in axes:
    ax.set_ylabel("Task accuracy (%)")
    ax.set_ylim(max(0, lo - 5), 100)
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
ax_c.legend(fontsize=8)
ax_both.legend(fontsize=6.5, ncol=1)
fig.suptitle("AwA2 complete — 50-class target, 85 concepts, test split", y=1.02)
fig.tight_layout()
plt.show()
